# shittyLLM: Basic BPE Tokenizer + Decoder-Only GPT Transformer

In [ ]:
import torch
from tokenizer import BasicTokenizer
from model import GPTLanguageModel

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"Using device: {device}")

In [ ]:
# 1. Load dataset & train BPE tokenizer
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

vocab_size = 300
tokenizer = BasicTokenizer()
tokenizer.train(text[:50000], vocab_size=vocab_size, verbose=False)
print(f"Tokenizer trained with vocab_size={vocab_size}")

# 2. Encode text
data = torch.tensor(tokenizer.encode(text), dtype=torch.long)
print(f"Total tokens: {len(data)}")

In [ ]:
# 3. Initialize Model
model = GPTLanguageModel(
    vocab_size=vocab_size,
    n_embd=128,
    n_head=4,
    n_layer=4,
    block_size=256,
    dropout=0.2,
    device=device
).to(device)

print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")

In [ ]:
# Test generation before training
prompt = "ROMEO:"
start_ids = tokenizer.encode(prompt)
context = torch.tensor([start_ids], dtype=torch.long, device=device)
out_ids = model.generate(context, max_new_tokens=50)[0].tolist()
print(tokenizer.decode(out_ids))